# CANDLE V3 — audited DeNICE V1 baseline configuration

Stage: v1_clone. Uses an isolated, byte-identical copy of the pinned V1 runtime.
Peer selection is unchanged. Complementary class selection is NOT implemented yet.
Fresh seed42; 6 tasks x20 rounds; batch2048; Adam lr.001; paper neighborhoods xi=.8.
One local model/router per client; binary_cosine during training; AMP enabled as in V1.
Historical content-disjoint BASE split, not 100% original TRAIN. No replay/APPLIANCE/CME.
Every round is checkpointed/compressed; each task seals one ZIP; 12 GiB output guard.
After final Task5/Round19 and terminal consolidation: full 34-class TEST on disjoint receiver shards.
Report BinarySelf as primary and frozen MulticlassSelf as secondary, without retraining the backbone.
Enable Internet and GPU. Only one GPU is used to preserve the V1 execution path.
Source is cloned from a pinned Git commit; no embedded source bundle/checkpoints/data.
Reuse mounted historical roles by setting ROLES_DIR; otherwise role preparation may take time.
Same configuration does not promise identical numerical results or an untouched confirmation test.


In [ ]:
# CANDLE V3: historical DeNICE V1 configuration; edit paths only.
import os
from pathlib import Path
import subprocess
import sys

CODE_REF = "d53e16ae9a3e55104106d94c4572c542fb848055"
DATA_DIR = "/kaggle/input/datasets/khoilv2005/100-clients"
OUTPUT_DIR = "/kaggle/working/results_candle_v3_xi_0p8_seed_42"
# Point to mounted historical role_lock.json's DIRECTORY to reuse the exact split.
# An empty directory here recreates roles from original TRAIN using split seed20261006.
ROLES_DIR = "/kaggle/working/candle_v3_roles"

env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
code = Path("/kaggle/temp/candle_v3_source")
code.parent.mkdir(parents=True, exist_ok=True)
if not code.exists():
    subprocess.check_call(["git", "clone", "--filter=blob:none", "--no-checkout",
        "https://github.com/khoilv2005/FL_IL_IDS.git", str(code)], env=env)
    subprocess.check_call(["git", "sparse-checkout", "set", "--no-cone",
        "/candle_v3_runtime/", "/tools/", "/configs/",
        "/requirements-candle-v3-kaggle.txt"], cwd=code, env=env)
    subprocess.check_call(["git", "checkout", "--detach", CODE_REF], cwd=code, env=env)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=code, text=True).strip()
if actual != CODE_REF:
    raise ValueError("Source differs from pinned CANDLE V3 revision; use a fresh session")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
    str(code / "requirements-candle-v3-kaggle.txt")])
print("CANDLE V3 supporting source:", actual, flush=True)
subprocess.check_call([sys.executable, "-u", str(code / "tools/train_candle_v3.py"),
    "--data-dir", DATA_DIR, "--output", OUTPUT_DIR, "--roles", ROLES_DIR], cwd=code, env=env)
print(Path(OUTPUT_DIR, "candle_v3_completion.json").read_text(encoding="utf-8"))
